# DSCubed Kaggle Competition 2026 
**Chi Nguyen**

## V1 Strategy
1. Predict price_change = p50 - p40 (regression)
2. Rank ascending (most negative = biggest drop = sell first)
3. Sell top N stocks with negative predictions, cap at N_SELLS=4000 
4. Model order: XGB v1 -> XGB weighted -> LGB v1 -> ensemble if OOF improves

In [189]:
# IMPORTS
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error
import warnings
import time
import os

warnings.filterwarnings('ignore')
start_time = time.time()

In [190]:
# CONFIG
TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
SUB_PATH = 'sample_submission.csv'

RANDOM_SEED = 67
N_FOLDS = 5
HOLDOUT_RATIO = 0.07

# Read sell quota from sample_submission after loading
SELL_RATIO = None  
N_SELLS = None  

In [191]:
# LOAD DATA
print("LOADING DATA")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SUB_PATH)

# Price columns are p1, p2, ..., p40, with p50 as the target
price_cols_all = sorted([c for c in train.columns if c.startswith('p') and c[1:].isdigit()],key=lambda x: int(x[1:]))
TARGET_COL ='p50'
PRICE_COLS = [c for c in price_cols_all if c != TARGET_COL]
LAST_PRICE = PRICE_COLS[-1]  # p40

# Derive constraint from sample submission template
N_SELLS = int(sub['sell'].sum())
SELL_RATIO = N_SELLS / len(sub)

print(f"Train: {train.shape[0]:,} rows × {train.shape[1]} cols")
print(f"Test: {test.shape[0]:,} rows × {test.shape[1]} cols")
print(f"Price cols: {PRICE_COLS[0]} → {PRICE_COLS[-1]} ({len(PRICE_COLS)} steps)")
print(f"N_SELLS: {N_SELLS:,} ({SELL_RATIO:.1%} of {len(sub):,} test stocks)")
print(f"Missing train: {train.isnull().sum().sum()}")
print(f"Missing test: {test.isnull().sum().sum()}")

assert set(test['ID']) == set(sub['ID']), "ID mismatch between test and submission."
print(f"ID check: All {len(sub):,} test IDs in sample_submission")

# Target: negative means price drops from p40 to p50
train['price_change'] = train[TARGET_COL] - train[LAST_PRICE]

pc = train['price_change']
print(f"\nTarget (price_change = p50–p40):")
print(f"  mean={pc.mean():.4f}, std={pc.std():.4f}, % drops={(pc<0).mean()*100:.1f}%")

# KFold is acceptable if there is no explicit time/session column
TIME_CANDIDATES = ['date', 'day', 'session', 'time_id', 'timestamp', 'ts_id']
time_cols = [c for c in train.columns if c.lower() in TIME_CANDIDATES]
if time_cols:
    print(f"\nTIME COLUMNS FOUND: {time_cols}")
    print(f"Consider switching to TimeSeriesSplit. Sort by time column first.")
else:
    print(f"\nNo time columns. KFold is valid.")

print(f"\nLoad time: {time.time()-start_time:.1f}s")

LOADING DATA
Train: 10,000 rows × 42 cols
Test: 10,000 rows × 41 cols
Price cols: p1 → p40 (40 steps)
N_SELLS: 4,000 (40.0% of 10,000 test stocks)
Missing train: 0
Missing test: 0
ID check: All 10,000 test IDs in sample_submission

Target (price_change = p50–p40):
  mean=0.0427, std=15.1770, % drops=50.2%

No time columns. KFold is valid.

Load time: 0.1s


In [192]:
# HOLDOUT SPLIT
# Keep a small holdout set separate from CV
# This is only used as a final sanity check after model training
train_main, train_holdout = train_test_split(train, test_size=HOLDOUT_RATIO, random_state=RANDOM_SEED)
y_main    = train_main['price_change']
y_holdout = train_holdout['price_change']

n_sells_main    = int(len(y_main) * SELL_RATIO)
n_sells_holdout = int(len(y_holdout) * SELL_RATIO)

print(f"Train main: {len(train_main):,} stocks")
print(f"Holdout:    {len(train_holdout):,} stocks")

Train main: 9,299 stocks
Holdout:    701 stocks


## Feature Engineering

Build features from `p1`–`p40` to describe price level, recent movement, trend, volatility

`v1` is the main feature set. `v2` and `v3` add extra technical indicators and are only used if OOF profit improves

In [193]:
def features_v1(df, price_cols):
    """Core feature set from p1-p40"""
    f = pd.DataFrame(index=df.index)
    prices = df[price_cols].values
    returns = np.diff(prices, axis=1)
    n = prices.shape[1]

    # Price levels
    f['last_price'] = prices[:, -1]
    f['first_price'] = prices[:, 0]
    f['price_mean'] = prices.mean(axis=1)
    f['price_std'] = prices.std(axis=1)
    f['price_range'] = prices.max(axis=1) - prices.min(axis=1)

    # Overall and recent percentage change
    f['return_total'] = (prices[:, -1] - prices[:, 0])  / (prices[:, 0] + 1e-8)
    f['return_last_5'] = (prices[:, -1] - prices[:, -6]) / (prices[:, -6] + 1e-8)
    f['return_last_10'] = (prices[:, -1] - prices[:, -11])/ (prices[:, -11] + 1e-8)

    # Step by step price changes
    f['mean_return'] = returns.mean(axis=1)
    f['std_return'] = returns.std(axis=1)
    f['last_return'] = returns[:, -1]
    f['last_3_return_mean'] = returns[:, -3:].mean(axis=1)
    f['last_5_return_mean'] = returns[:, -5:].mean(axis=1)

    # Moving averages 
    for w in [5, 10, 20]:
        ma = prices[:, -w:].mean(axis=1)
        f[f'ma_{w}'] = ma
        f[f'price_vs_ma_{w}'] = prices[:, -1] - ma

    # MA crossovers
    ma5 = prices[:, -5:].mean(axis=1)
    ma10 = prices[:, -10:].mean(axis=1)
    ma20 = prices[:, -20:].mean(axis=1)
    
    f['ma_cross_5_10'] = ma5 - ma10
    f['ma_cross_5_20'] = ma5 - ma20
    f['ma_cross_10_20'] = ma10 - ma20

    # Volatility
    f['vol_full']    = prices.std(axis=1)
    f['vol_last_10'] = prices[:, -10:].std(axis=1)
    f['vol_last_5']  = prices[:, -5:].std(axis=1)

    # Direction balance
    f['up_ratio'] = (returns > 0).sum(axis=1) / (n - 1)

    f = f.replace([np.inf, -np.inf], np.nan).fillna(0)
    return f


def features_v2_add(df, price_cols):
    """Extra trend, momentum, range-position features"""
    f = pd.DataFrame(index=df.index)
    
    prices = df[price_cols].values
    returns = np.diff(prices, axis=1)

    # Linear trend over recent windows
    for w in [5, 10, 20, 40]:
        f[f'slope_{w}'] = np.array([
            np.polyfit(np.arange(w), prices[i, -w:], 1)[0]
            for i in range(len(prices))
        ])

    # Momentum acceleration
    f['mom_change'] = returns[:, -5:].mean(axis=1) - returns[:, -10:-5].mean(axis=1)
    f['mom_change_long'] = returns[:, -10:].mean(axis=1) - returns[:, -20:-10].mean(axis=1)

    # Position of p40 within recent price range
    for w in [10, 20, 40]:
        hi = prices[:, -w:].max(axis=1)
        lo = prices[:, -w:].min(axis=1)
        f[f'position_{w}'] = (prices[:, -1] - lo) / (hi - lo + 1e-8)

    # Distance from recent extremes
    f['dist_from_high_10'] = prices[:, -1] - prices[:, -10:].max(axis=1)
    f['dist_from_low_10'] = prices[:, -1] - prices[:, -10:].min(axis=1)
    f['dist_from_high_40'] = prices[:, -1] - prices.max(axis=1)
    f['dist_from_low_40']  = prices[:, -1] - prices.min(axis=1)

    # Recent direction count
    f['recent_up_count'] = (returns[:, -5:] > 0).sum(axis=1)
    f['recent_down_count'] = (returns[:, -5:] < 0).sum(axis=1)

    # Drawdown from running peak
    cummax = np.maximum.accumulate(prices, axis=1)
    drawdowns = (prices - cummax) / (cummax + 1e-8)
    f['max_drawdown'] = drawdowns.min(axis=1)

    # Large within-series move flag
    return_std = returns.std(axis=1)
    return_mean = returns.mean(axis=1)
    f['has_extreme_move'] = (np.abs(returns).max(axis=1) > return_mean + 2 * return_std).astype(int)

    f = f.replace([np.inf, -np.inf], np.nan).fillna(0)
    return f


def features_v3_add(df, price_cols):
    """Extra technical indicators: log returns, RSI, Bollinger, Exponential MA, autocorr"""
    f = pd.DataFrame(index=df.index)
    prices = df[price_cols].values
    n  = prices.shape[1]

    # Log returns
    log_ret = np.log(prices[:, 1:] / (prices[:, :-1] + 1e-8))
    f['log_return_mean'] = log_ret.mean(axis=1)
    f['log_return_std'] = log_ret.std(axis=1)
    f['log_return_skew'] = pd.DataFrame(log_ret).skew(axis=1).values
    f['log_return_kurt'] = pd.DataFrame(log_ret).kurtosis(axis=1).values

    # RSI style momentum
    gains = np.where(log_ret > 0, log_ret, 0)
    losses = np.where(log_ret < 0, -log_ret, 0)
    for w in [5, 10, 14]:
        rs = gains[:, -w:].mean(axis=1) / (losses[:, -w:].mean(axis=1) + 1e-8)
        f[f'rsi_{w}'] = 100 - (100 / (1 + rs))

    # Bollinger band position
    for w in [10, 20]:
        ma  = prices[:, -w:].mean(axis=1)
        std = prices[:, -w:].std(axis=1)
        f[f'bb_upper_{w}'] = (prices[:, -1] - (ma + 2*std)) / (std + 1e-8)
        f[f'bb_lower_{w}'] = (prices[:, -1] - (ma - 2*std)) / (std + 1e-8)

    # Exponential MA
    for span in [5, 10, 20]:
        alpha = 2 / (span + 1)
        ema = prices[:, 0].copy()
        for t in range(1, n):
            ema = alpha * prices[:, t] + (1 - alpha) * ema
        f[f'ema_{span}']          = ema
        f[f'price_vs_ema_{span}'] = prices[:, -1] - ema

    # Autocorrelation of returns
    returns = np.diff(prices, axis=1)
    for lag in [1, 3, 5]:
        corrs = []
        for i in range(len(returns)):
            c = np.corrcoef(returns[i, :-lag], returns[i, lag:])[0, 1] if len(returns[i]) > lag else 0
            corrs.append(0 if np.isnan(c) else c)
        f[f'autocorr_lag_{lag}'] = corrs

    # Price ratios to recent lags 
    for lag in [1, 2, 3, 5, 10]:
        f[f'price_ratio_lag_{lag}'] = prices[:, -1] / (prices[:, -(lag+1)] + 1e-8)

    f = f.replace([np.inf, -np.inf], np.nan).fillna(0)
    return f


def build_features(df, price_cols, version='v1'):
    """Build selected feature set"""
    print(f"  Building features ({version})...", end=' ')
    f = features_v1(df, price_cols)
    if version in ('v2', 'v3'):
        f = pd.concat([f, features_v2_add(df, price_cols)], axis=1)
    if version == 'v3':
        f = pd.concat([f, features_v3_add(df, price_cols)], axis=1)
    print(f"{f.shape[1]} features")
    return f

---
## Evaluation Functions

In [194]:
def make_sell_decisions(predictions, n_sells):
    """
    Sell up to n_sells stocks with the lowest predicted price_change
    Only sells stocks predicted to drop (prediction < 0)
    Sign: price_change = p50 - p40
    """
    sell = np.zeros(len(predictions))
    drop_idx = np.where(predictions < 0)[0]
    if len(drop_idx) == 0:
        return sell.astype(int)
    ranked = np.argsort(predictions[drop_idx]) # most negative first
    n_to_sell = min(len(drop_idx), n_sells)
    sell[drop_idx[ranked[:n_to_sell]]] = 1
    return sell.astype(int)


def calculate_profit(y_true, sell):
    """Profit from selling before a price drop"""
    return float(np.sum(sell * (-y_true)))


def evaluate(y_true, preds, n_sells, label=''):
    """Evaluate predictions using profit, MAE, sell accuracy"""
    sell = make_sell_decisions(preds, n_sells)
    profit = calculate_profit(y_true, sell)
    mae = mean_absolute_error(y_true, preds)
    n_sell = int(sell.sum())
    acc = np.sum(sell * (y_true < 0)) / n_sell if n_sell > 0 else 0
    print(f"{label:25s} | Profit: {profit:9.2f} | MAE: {mae:.4f} | "f"Sell acc: {acc:.1%} | Sells: {n_sell}/{n_sells}")
    return profit


def validate_sign(y_true, preds, n_sells, label=''):
    """Check that sell decisions are based on lower predicted price_change"""
    sell  = make_sell_decisions(preds, n_sells)
    profit = calculate_profit(y_true, sell)
    pred_drops = (preds < 0).mean()
    true_drops = (y_true < 0).mean()

    print(f"\nSign check ({label})")
    print(f"Predicted drops: {pred_drops:.1%} | Actual drops: {true_drops:.1%}")
    print(f"Profit: ${profit:.2f}")

    if profit < 0:
        print(f"LOSS. Check sign logic immediately")
    else:
        print(f"Positive profit. Good :)")

    if abs(pred_drops - true_drops) > 0.15:
        print(f"Large divergence. Possible sign error or distribution shift")
    return profit


def submission_sign_check(predictions, sell_decisions):
    """Check prediction means for sell and hold groups before saving"""
    df = pd.DataFrame({'pred': predictions, 'sell': sell_decisions})
    grp = df.groupby('sell')['pred'].mean()
    print(f"\nPrediction distribution check:")
    print(f"HOLD (sell=0) mean: {grp.get(0, float('nan')):.4f}")
    print(f"SELL (sell=1) mean: {grp.get(1, float('nan')):.4f}")
    if 0 in grp.index and 1 in grp.index:
        if grp[1] < grp[0]:
            print(f"Correct. SELL group has lower/more negative predictions :)")
        else:
            print(f"SIGN ERROR. SELL group has HIGHER predictions than HOLD. DON'T SUBMIT :(")


def create_submission(test_df, predictions, n_sells, filename):
    """Make submission CSV"""
    sell = make_sell_decisions(predictions, n_sells)

    if int(sell.sum()) > n_sells: # Trim safety
        idx  = np.where(sell == 1)[0]
        keep = idx[np.argsort(predictions[idx])[:n_sells]]
        sell = np.zeros(len(sell), dtype=int)
        sell[keep] = 1

    submission_sign_check(predictions, sell)

    sub_df = pd.DataFrame({'ID': test_df['ID'].values, 'sell': sell})
    sub_df.to_csv(filename, index=False)
    print(f"  Saved: {filename} | Sells: {int(sell.sum())}/{n_sells} | "
          f"Ratio: {sell.mean():.1%} | "
          f"{'PASS' if int(sell.sum()) <= n_sells else 'FAIL'}")
    return sub_df

---
## Model Training
OOF predictions are used to compare models on training data. Test predictions are averaged across folds for submission

In [195]:
def train_xgb(X_tr_all, y_tr_all, X_test, n_folds=N_FOLDS, seed=RANDOM_SEED,
              sample_weights=None, label='XGB'):
    """
    Train XGBoost with K-Fold CV

    Returns:
    - OOF predictions for model evaluation
    - averaged test predictions for submission
    - fitted fold models
    """
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")

    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'n_estimators': 1000,
        'early_stopping_rounds': 50,
        'verbosity': 0,
        'random_state': seed,
    }

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_tr_all))
    tst_preds = np.zeros(len(X_test))
    models = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr_all)):
        X_tr, X_val = X_tr_all.iloc[tr_idx], X_tr_all.iloc[val_idx]
        y_tr, y_val = y_tr_all.iloc[tr_idx], y_tr_all.iloc[val_idx]

        w = sample_weights[tr_idx] if sample_weights is not None else None

        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, sample_weight=w,
                  eval_set=[(X_val, y_val)], verbose=False)

        oof_preds[val_idx] = model.predict(X_val)
        tst_preds += model.predict(X_test) / n_folds
        models.append(model)

    n_sells_oof = int(len(y_tr_all) * SELL_RATIO)
    evaluate(y_tr_all.values, oof_preds, n_sells_oof, f"{label} OOF")
    validate_sign(y_tr_all.values, oof_preds, n_sells_oof, label)
    return oof_preds, tst_preds, models


def train_lgb(X_tr_all, y_tr_all, X_test, n_folds=N_FOLDS, seed=RANDOM_SEED):
    """Train LightGBM with K-Fold CV"""
    print(f"\n{'='*60}")
    print("LIGHTGBM")
    print(f"{'='*60}")

    params = {
        'objective': 'regression', 'metric': 'mae',
        'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
        'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.1, 'reg_lambda': 0.1, 'n_estimators': 1000,
        'verbose': -1, 'random_state': seed,
    }

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_tr_all))
    tst_preds = np.zeros(len(X_test))
    models = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr_all)):
        X_tr, X_val = X_tr_all.iloc[tr_idx], X_tr_all.iloc[val_idx]
        y_tr, y_val = y_tr_all.iloc[tr_idx], y_tr_all.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

        oof_preds[val_idx] = model.predict(X_val)
        tst_preds         += model.predict(X_test) / n_folds
        models.append(model)

    n_sells_oof = int(len(y_tr_all) * SELL_RATIO)
    evaluate(y_tr_all.values, oof_preds, n_sells_oof, "LGB OOF")
    validate_sign(y_tr_all.values, oof_preds, n_sells_oof, "LGB")

    imp = pd.DataFrame({
        'feature': X_tr_all.columns,
        'importance': np.mean([m.feature_importances_ for m in models], axis=0)
    }).sort_values('importance', ascending=False)
    print(f"\nTop 10 features:")
    for _, row in imp.head(10).iterrows():
        print(f"{row['feature']:30s} {row['importance']:.0f}")

    return oof_preds, tst_preds, models


def make_sample_weights(y, clip_quantile=0.95):
    """
    Create sample weights from absolute price_change.

    Larger moves receive higher weight because they matter more for profit.
    Weights are clipped to reduce outlier influence and normalised to mean 1.
    """
    w = np.abs(y)
    cap = np.quantile(w, clip_quantile)
    w = np.clip(w, 0, cap)
    w = w / (w.mean() + 1e-8)
    return w

---
## Run Training Pipeline
Build the selected feature set, track model scores, run baseline checks before training the main models

In [196]:
print("BUILDING FEATURES")

X_train = build_features(train_main, PRICE_COLS, version='v1')
X_test  = build_features(test, PRICE_COLS, version='v1')
X_hold  = build_features(train_holdout, PRICE_COLS, version='v1')

print(f"\nBuild time: {time.time()-start_time:.1f}s")

BUILDING FEATURES
  Building features (v1)... 26 features
  Building features (v1)... 26 features
  Building features (v1)... 26 features

Build time: 0.1s


In [197]:
# MODEL SCORES TRACKER 
# Only submit if model beats max(model_scores)!
model_scores = {}

In [198]:
print("BASELINES")
# Simple benchmarks before model training, the trained models should beat these
evaluate(y_main.values, -X_train['last_5_return_mean'].values, n_sells_main, "Momentum")
evaluate(y_main.values, np.zeros(len(y_main)), n_sells_main, "Zero (random sell)")

BASELINES
Momentum                  | Profit: -10247.54 | MAE: 12.8768 | Sell acc: 42.8% | Sells: 3719/3719
Zero (random sell)        | Profit:      0.00 | MAE: 12.5188 | Sell acc: 0.0% | Sells: 0/3719


0.0

In [199]:
# XGB v1
xgb_oof, xgb_tst, xgb_models = train_xgb(X_train, y_main, X_test, label='XGB v1')

model_scores['xgb_v1'] = calculate_profit(y_main.values, make_sell_decisions(xgb_oof, n_sells_main))
print(f"\nmodel_scores: {model_scores}")

print("\nCreating submission #1: XGB v1")
create_submission(test, xgb_tst, N_SELLS, 'submission_xgb_v1.csv')
print(f"Total time: {time.time()-start_time:.1f}s")


XGB v1
XGB v1 OOF                | Profit:  18278.08 | MAE: 11.9429 | Sell acc: 63.4% | Sells: 3719/3719

Sign check (XGB v1)
Predicted drops: 48.2% | Actual drops: 50.2%
Profit: $18278.08
Positive profit. Good :)

model_scores: {'xgb_v1': 18278.079999999998}

Creating submission #1: XGB v1

Prediction distribution check:
HOLD (sell=0) mean: 3.1583
SELL (sell=1) mean: -4.3366
Correct. SELL group has lower/more negative predictions :)
  Saved: submission_xgb_v1.csv | Sells: 4000/4000 | Ratio: 40.0% | PASS
Total time: 0.8s


In [200]:
# XGB v1 WEIGHTED
# Weight larger price changes more heavily during training.
# Submit only if OOF profit improves over the unweighted XGB model.
xgb_w_oof, xgb_w_tst, xgb_w_models = train_xgb(
    X_train, y_main, X_test,
    sample_weights=make_sample_weights(y_main.values),
    label='XGB v1 Weighted')

model_scores['xgb_v1_weighted'] = calculate_profit(y_main.values, make_sell_decisions(xgb_w_oof, n_sells_main))
print(f"\nmodel_scores: {model_scores}")

if model_scores['xgb_v1_weighted'] > model_scores['xgb_v1']:
    print(f"Weighted beats unweighted. Creating submission.")
    create_submission(test, xgb_w_tst, N_SELLS, 'submission_xgb_v1_weighted.csv')
else:
    print(f"Weighted did not improve. Skipping submission.")


XGB v1 Weighted
XGB v1 Weighted OOF       | Profit:  17732.23 | MAE: 11.9239 | Sell acc: 63.1% | Sells: 3719/3719

Sign check (XGB v1 Weighted)
Predicted drops: 48.1% | Actual drops: 50.2%
Profit: $17732.23
Positive profit. Good :)

model_scores: {'xgb_v1': 18278.079999999998, 'xgb_v1_weighted': 17732.23}
Weighted did not improve. Skipping submission.


In [201]:
# LGB v1
lgb_oof, lgb_tst, lgb_models = train_lgb(X_train, y_main, X_test)

model_scores['lgb_v1'] = calculate_profit(y_main.values, make_sell_decisions(lgb_oof, n_sells_main))
print(f"\nmodel_scores: {model_scores}")

if model_scores['lgb_v1'] > max(model_scores.values()):
    print(f"LGB beats current best. Creating submission.")
    create_submission(test, lgb_tst, N_SELLS, 'submission_lgb_v1.csv')
else:
    print(f"LGB profit: ${model_scores['lgb_v1']:.2f} (not best)")


LIGHTGBM
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[32]	valid_0's l1: 11.884
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[39]	valid_0's l1: 11.8108
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[44]	valid_0's l1: 11.7035
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[57]	valid_0's l1: 12.0745
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[48]	valid_0's l1: 12.2513
LGB OOF                   | Profit:  18343.80 | MAE: 11.9448 | Sell acc: 63.4% | Sells: 3719/3719

Sign check (LGB)
Predicted drops: 48.3% | Actual drops: 50.2%
Profit: $18343.80
Positive profit. Good :)

Top 10 features:
last_return                    213
first_price                    95
std_return                     68
vol_last_5                     63
vol_last_10      

In [202]:
# ENSEMBLE v1
ens_oof_v1 = (xgb_oof + lgb_oof) / 2
ens_tst_v1 = (xgb_tst + lgb_tst) / 2

model_scores['ensemble_v1'] = calculate_profit(
    y_main.values, make_sell_decisions(ens_oof_v1, n_sells_main)
)
evaluate(y_main.values, ens_oof_v1, n_sells_main, 'Ensemble v1 OOF')
print(f"\nmodel_scores: {model_scores}")

if model_scores['ensemble_v1'] > max({k: v for k, v in model_scores.items() if k != 'ensemble_v1'}.values()):
    print(f"✓ Ensemble beats all single models. Creating submission.")
    validate_sign(y_main.values, ens_oof_v1, n_sells_main, "Ensemble v1")
    create_submission(test, ens_tst_v1, N_SELLS, 'submission_ensemble_v1.csv')
else:
    print(f"Ensemble does not beat best single model. Skipping.") # Don't force ensemble just because 

print(f"\nTime so far: {time.time()-start_time:.1f}s")

Ensemble v1 OOF           | Profit:  18245.80 | MAE: 11.9354 | Sell acc: 63.3% | Sells: 3719/3719

model_scores: {'xgb_v1': 18278.079999999998, 'xgb_v1_weighted': 17732.23, 'lgb_v1': 18343.800000000003, 'ensemble_v1': 18245.800000000003}
Ensemble does not beat best single model. Skipping.

Time so far: 2.7s


In [203]:
# === v3 FEATURES ===
# Submit v3 only if it improves OOF profit

print("# FEATURE UPGRADE: v3")

X_train_v3 = build_features(train_main, PRICE_COLS, version='v3')
X_test_v3 = build_features(test, PRICE_COLS, version='v3')
X_hold_v3 = build_features(train_holdout, PRICE_COLS, version='v3')

xgb_oof_v3, xgb_tst_v3, xgb_models_v3 = train_xgb(X_train_v3, y_main, X_test_v3, label='XGB v3')
lgb_oof_v3, lgb_tst_v3, lgb_models_v3 = train_lgb(X_train_v3, y_main, X_test_v3)
ens_oof_v3 = (xgb_oof_v3 + lgb_oof_v3) / 2
ens_tst_v3 = (xgb_tst_v3 + lgb_tst_v3) / 2

model_scores['xgb_v3'] = calculate_profit(y_main.values, make_sell_decisions(xgb_oof_v3, n_sells_main))
model_scores['lgb_v3'] = calculate_profit(y_main.values, make_sell_decisions(lgb_oof_v3, n_sells_main))
model_scores['ensemble_v3'] = calculate_profit(y_main.values, make_sell_decisions(ens_oof_v3, n_sells_main))

print(f"\nFull model_scores: {model_scores}")
current_best = max(model_scores, key=model_scores.get)
print(f"Current best: {current_best} = ${model_scores[current_best]:.2f}")

v1_best = max(model_scores.get('xgb_v1', 0), model_scores.get('lgb_v1', 0),
              model_scores.get('ensemble_v1', 0), model_scores.get('xgb_v1_weighted', 0))

if model_scores['xgb_v3'] > v1_best:
    print(f"XGB v3 beats v1. Submitting.")
    validate_sign(y_main.values, xgb_oof_v3, n_sells_main, "XGB v3")
    create_submission(test, xgb_tst_v3, N_SELLS, 'submission_xgb_v3.csv')

if model_scores['ensemble_v3'] > v1_best:
    print(f"Ensemble v3 beats v1. Submitting.")
    validate_sign(y_main.values, ens_oof_v3, n_sells_main, "Ensemble v3")
    create_submission(test, ens_tst_v3, N_SELLS, 'submission_ensemble_v3.csv')

if model_scores['xgb_v3'] <= v1_best and model_scores['ensemble_v3'] <= v1_best:
    print(f"v3 features did not improve. Stick with v1.")

# FEATURE UPGRADE: v3
  Building features (v3)... 68 features
  Building features (v3)... 68 features
  Building features (v3)... 68 features

XGB v3
XGB v3 OOF                | Profit:  18234.18 | MAE: 11.9684 | Sell acc: 63.3% | Sells: 3719/3719

Sign check (XGB v3)
Predicted drops: 48.4% | Actual drops: 50.2%
Profit: $18234.18
Positive profit. Good :)

LIGHTGBM
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[27]	valid_0's l1: 11.9222
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[37]	valid_0's l1: 11.822
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's l1: 11.6804
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[58]	valid_0's l1: 12.1275
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[58]	valid_0's l1: 12.2109
LGB OOF            

In [204]:
# HOLDOUT EVALUATION
# Evaluate selected models on the reserved holdout set (just for sanity check)
print("HOLDOUT EVALUATION (unseen data for sanity check only)")

for models_list, X_ho, label in [
    (xgb_models, X_hold, 'XGB v1'),
    (lgb_models, X_hold, 'LGB v1'),
    (xgb_models_v3, X_hold_v3, 'XGB v3'),
]:
    preds_ho = np.mean([m.predict(X_ho) for m in models_list], axis=0)
    evaluate(y_holdout.values, preds_ho, n_sells_holdout, f"{label} HOLDOUT")

HOLDOUT EVALUATION (unseen data for sanity check only)
XGB v1 HOLDOUT            | Profit:   1440.82 | MAE: 12.3268 | Sell acc: 62.1% | Sells: 280/280
LGB v1 HOLDOUT            | Profit:   1435.24 | MAE: 12.2912 | Sell acc: 62.9% | Sells: 280/280
XGB v3 HOLDOUT            | Profit:   1560.51 | MAE: 12.3152 | Sell acc: 63.9% | Sells: 280/280


In [205]:
# N_SELLS SENSITIVITY 
# Test different sell counts using the best OOF model. Final submission uses the sell count with highest OOF profit 
print("N_SELLS SENSITIVITY (best OOF model)")

best_name = max(model_scores, key=model_scores.get)
print(f"Running on: {best_name}")

_oof_map = {
    'xgb_v1': xgb_oof, 'xgb_v1_weighted': xgb_w_oof,
    'lgb_v1': lgb_oof, 'ensemble_v1': ens_oof_v1,
    'xgb_v3': xgb_oof_v3, 'lgb_v3': lgb_oof_v3, 'ensemble_v3': ens_oof_v3,}
_tst_map = {
    'xgb_v1': xgb_tst, 'xgb_v1_weighted': xgb_w_tst,
    'lgb_v1': lgb_tst, 'ensemble_v1': ens_tst_v1,
    'xgb_v3': xgb_tst_v3, 'lgb_v3': lgb_tst_v3, 'ensemble_v3': ens_tst_v3,}
best_oof_preds = _oof_map[best_name]
best_tst_preds = _tst_map[best_name]

best_profit, best_n = -np.inf, N_SELLS
for ratio in np.arange(0.10, 0.55, 0.05):
    n      = min(int(len(y_main) * ratio), N_SELLS)
    sell   = make_sell_decisions(best_oof_preds, n)
    profit = calculate_profit(y_main.values, sell)
    marker = " BEST" if profit > best_profit else ""
    if profit > best_profit:
        best_profit, best_n = profit, int(sell.sum())
    print(f"  N={n:6d} ({ratio:.0%}) | Profit: {profit:10.2f}{marker}")

final_n = min(best_n, N_SELLS)
print(f"\nOptimal N: {final_n} (ratio: {final_n/len(y_main):.1%})")
create_submission(test, best_tst_preds, final_n, 'submission_final.csv')

N_SELLS SENSITIVITY (best OOF model)
Running on: ensemble_v3
  N=   929 (10%) | Profit:    5697.32 BEST
  N=  1394 (15%) | Profit:    9066.21 BEST
  N=  1859 (20%) | Profit:   11303.38 BEST
  N=  2324 (25%) | Profit:   13935.24 BEST
  N=  2789 (30%) | Profit:   16130.72 BEST
  N=  3254 (35%) | Profit:   17543.59 BEST
  N=  3719 (40%) | Profit:   18421.00 BEST
  N=  4000 (45%) | Profit:   18598.93 BEST
  N=  4000 (50%) | Profit:   18598.93

Optimal N: 4000 (ratio: 43.0%)

Prediction distribution check:
HOLD (sell=0) mean: 3.0964
SELL (sell=1) mean: -4.2766
Correct. SELL group has lower/more negative predictions :)
  Saved: submission_final.csv | Sells: 4000/4000 | Ratio: 40.0% | PASS


,ID,sell
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
...,...,...
9995,9995,0
9996,9996,0
9997,9997,1
9998,9998,1


In [206]:
# FINAL SUMMARY 
print("FINAL MODEL LEADERBOARD")
for name, profit in sorted(model_scores.items(), key=lambda x: x[1], reverse=True):
    marker = " BEST" if name == max(model_scores, key=model_scores.get) else ""
    print(f"  {name:25s}: ${profit:10.2f}{marker}")

print(f"\nSubmissions created:")
for f in sorted([f for f in os.listdir('.') if f.startswith('submission')]):
    print(f"{f}")

print(f"\nTotal time: {time.time()-start_time:.1f}s")
print(f"N_SELLS = {N_SELLS} | SELL_RATIO = {SELL_RATIO:.1%}")
print(f"\nSelect best submission based on model_scores above")

FINAL MODEL LEADERBOARD
  ensemble_v3              : $  18421.00 BEST
  lgb_v1                   : $  18343.80
  xgb_v1                   : $  18278.08
  ensemble_v1              : $  18245.80
  lgb_v3                   : $  18242.92
  xgb_v3                   : $  18234.18
  xgb_v1_weighted          : $  17732.23

Submissions created:
submission_cv_ensemble.csv
submission_ensemble_v1.csv
submission_ensemble_v3.csv
submission_final.csv
submission_final_v2.csv
submission_xgb_full.csv
submission_xgb_v1.csv
submission_xgb_v1_weighted.csv
submission_xgb_v3.csv

Total time: 7.2s
N_SELLS = 4000 | SELL_RATIO = 40.0%

Select best submission based on model_scores above
